# paper-verdict-finetune -- Colab GPU runner

Before running: Runtime -> Change runtime type -> GPU.

This notebook is intentionally thin: it clones the repo, installs
requirements, and calls the scripts in `src/`. All logic lives there,
not here, so it stays versioned and testable outside Colab too.

In [ ]:
!nvidia-smi

In [ ]:
BRANCH = "verdict-finetune-architecture"
!git clone --branch $BRANCH --depth 1 https://github.com/AngryWatership/textor.git repo
%cd repo

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# sanity-check the split before committing GPU time to training
!python src/data.py --data data/papers.jsonl --split-out outputs/split.json

In [ ]:
BASE_MODEL = "allenai/specter2_base"  # or sentence-transformers/all-MiniLM-L6-v2, allenai/scibert_scivocab_uncased
FINETUNE_MODE = "partial"  # or "full"

!cd src && python train.py \
    --base-model $BASE_MODEL \
    --finetune-mode $FINETUNE_MODE \
    --data ../data/papers.jsonl \
    --split-out ../outputs/split.json \
    --out-dir ../outputs

In [ ]:
!cd src && python evaluate.py \
    --checkpoint ../outputs/best_model.pt \
    --base-model $BASE_MODEL \
    --finetune-mode $FINETUNE_MODE \
    --data ../data/papers.jsonl \
    --split-out ../outputs/split.json \
    --out-dir ../outputs

In [ ]:
import json
report = json.load(open("outputs/eval_report.json"))
print("confusion matrix (rows=true, cols=pred), labels:", report["confusion_matrix_labels"])
for row in report["confusion_matrix"]:
    print(row)
print("test silhouette:", report["test_silhouette_euclidean"])
print("test knn10 purity:", report["test_knn10_purity"])
print("per-class purity:", report["test_knn10_purity_per_class"])

In [ ]:
# optional: download the checkpoint + reports back to your machine
from google.colab import files
files.download("outputs/eval_report.json")
files.download("outputs/train_summary.json")
# files.download("outputs/best_model.pt")  # this is a few hundred MB, uncomment if you want it